## ACUTE MYELOID LUEKANEMIA PHENOTYPE FILE 
This script creates the phenotype file for SAIGE-GENE+. It integrates several inputs, including EUR-like exome sample quality control results, ancestry-specific PCs, and age and sex files. Only individuals who pass exome quality control are retained in the SAIGE-GENE+ phenotype file.
Case–control status is assigned by providing a list of person IDs for individuals designated as cases; all remaining individuals are assigned as controls. The script also restricts the dataset to individuals who have electronic health record data. All quantitative covariates are standardized to have a mean of 0 and a standard deviation of 1.
Because more samples passed exome quality control than ACAF quality control in this analysis, this phenotype file will also be used for the ACAF analysis, with samples failing ACAF quality control excluded at a later stage.

In [1]:
# Load packages 
from sklearn.preprocessing import StandardScaler
from functools import reduce
import pandas as pd 
import os 

In [4]:
%run /home/jupyter/workspace/rw-migration-aou-rw-9f879fc4/notebooks/Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: NOT FOUND
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/jupyter/.bashrc
C2024Q3R9 BQ_DATASET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = None
wb-silky-artichoke-2408.C2024Q3R9



In [5]:
# Workspace bucket 
my_bucket = 'gs://rw-migration-aou-rw-9f879fc4'

In [2]:
# Make directory for aml phenotypes files 
!rm -rf aml_phenotype_files
!mkdir -p aml_phenotype_files

In [ ]:
# Load quality control information and demographic information 
bucket = f"{my_bucket}/data/pcs/EUR_flashpca.pcs"
eur_pca = pd.read_csv(bucket, sep ='\t')

bucket = f"{my_bucket}/data/wgs_qc/analysis_samples_wgs_qc/exome/EUR-like_exome_qc.txt"
eur_exome = pd.read_csv(bucket, sep ='\t')

bucket = f"{my_bucket}/data/covariates/biosample_collection_age.tsv"
age = pd.read_csv(bucket, sep ='\t')

bucket = f"{my_bucket}/data/wgs_qc/ancestry_sd3_wgs_qc/EUR-like_cohort_info_qc.txt"
sex = pd.read_csv(bucket, sep ='\t')

In [ ]:
# Assign sex variable 
sex['sex'] = sex['dragen_sex_ploidy'].apply(lambda x: 1 if x == "XX" else 0 if x == "XY" else 2)
sex['sex'] = sex['sex'].astype("int")

In [7]:
import pandas
import os

# This query represents dataset "aml_cohort" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_39009800_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (140352)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (4304199) 
                    AND is_standard = 1 )) criteria ) )"""

aml_person_df = pandas.read_gbq(
    dataset_39009800_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

Downloading:   0%|          | 0/774 [00:00<?, ?rows/s]

In [8]:
import pandas
import os

# This query represents dataset "electronic_health_records" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_72216074_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

ehr_person_df = pandas.read_gbq(
    dataset_72216074_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

Downloading:   0%|          | 0/393601 [00:00<?, ?rows/s]

In [9]:
# Create list of quality control information and demographic information 
dfs = [eur_exome.rename(columns={"IID":"person_id"}),
       age.rename(columns={"IID":"person_id"}).filter(regex="person_id|age"),
       sex.rename(columns={"IID":"person_id"}).filter(regex="person_id|sex"), 
       eur_pca.rename(columns={"IID":"person_id"}).filter(regex="person_id|PC"),
       ehr_person_df
       ]

In [10]:
# Combine list of quality control information and demographic information 
phenotype_file = reduce(
    lambda left, right: pd.merge(left, right, on="person_id", how="inner"),
    dfs
)

In [11]:
# Assign AML phenotype to individuals 
phenotype_file['aml_biosample_age'] = phenotype_file['person_id'].isin(aml_person_df['person_id']).astype(int)

In [12]:
# Format phenotype file  
phenotype_file = (phenotype_file[['aml_biosample_age','sex', 'age', 
            'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 
            'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 
            'person_id']]
)

In [13]:
# Identify columns that start with "PC" or "age"
cols_to_standardize = [col for col in phenotype_file.columns if col.startswith("PC") or col.startswith("age")]
# Standardize only the selected columns
scaler = StandardScaler()
phenotype_file[cols_to_standardize] = scaler.fit_transform(phenotype_file[cols_to_standardize])
# Print results of the standarization 
print("============================================")
print("Check variances of standardized covariates:")
print("============================================")
print(phenotype_file.filter(regex="PC|age").var())
print(phenotype_file.describe())
print("============================================")

Check variances of standardized covariates:
aml_biosample_age    0.001557
age                  1.000006
PC1                  1.000006
PC2                  1.000006
PC3                  1.000006
PC4                  1.000006
PC5                  1.000006
PC6                  1.000006
PC7                  1.000006
PC8                  1.000006
PC9                  1.000006
PC10                 1.000006
dtype: float64
       aml_biosample_age            sex           age           PC1  \
count      169971.000000  169971.000000  1.699710e+05  1.699710e+05   
mean            0.001559       0.605039  4.173688e-16  1.672151e-18   
std             0.039455       0.488844  1.000003e+00  1.000003e+00   
min             0.000000       0.000000 -2.303439e+00 -1.182560e+00   
25%             0.000000       0.000000 -7.940227e-01 -4.671958e-01   
50%             0.000000       1.000000  1.821123e-01 -3.412527e-01   
75%             0.000000       1.000000  7.896915e-01 -1.072462e-01   
max          

In [14]:
# Final check for phenotype 
print("============================================")
print("Check the presence of aml phenotype:")
print("============================================")
print(phenotype_file["aml_biosample_age"].value_counts())
print("============================================")

Check the presence of aml phenotype:
aml_biosample_age
0    169706
1       265
Name: count, dtype: int64


In [15]:
# Save phenoytpe file
phenotype_file.to_csv("EUR_phenotype_file.txt", sep ='\t', index=None)

In [16]:
# Save phenoytpe file to workspace bucket 
!gsutil cp EUR_phenotype_file.txt {my_bucket}/data/saige_phenotype_files/aml_biosample_age_files/

Copying file://EUR_phenotype_file.txt [Content-Type=text/plain]...
- [1 files][ 37.1 MiB/ 37.1 MiB]    1.5 MiB/s                                   
Operation completed over 1 objects/37.1 MiB.                                     
